In [3]:
import numpy as np
import matplotlib.pyplot as plt
n=50
samples = np.arange(n) 
sampling_rate=100
wave_velocity=8000



#use this function to generate signal_A and signal_B with a random shift
def generate_signals(frequency=5):

    noise_freqs = [15, 30, 45]  # Default noise frequencies in Hz

    amplitudes = [0.5, 0.3, 0.1]  # Default noise amplitudes
    noise_freqs2 = [10, 20, 40] 
    amplitudes2 = [0.3, 0.2, 0.1]
    
     # Discrete sample indices
    dt = 1 / sampling_rate  # Sampling interval in seconds
    time = samples * dt  # Time points corresponding to each sample

    # Original clean signal (sinusoidal)
    original_signal = np.sin(2 * np.pi * frequency * time)

    # Adding noise
    noise_for_sigal_A = sum(amplitude * np.sin(2 * np.pi * noise_freq * time)
                for noise_freq, amplitude in zip(noise_freqs, amplitudes))
    noise_for_sigal_B = sum(amplitude * np.sin(2 * np.pi * noise_freq * time)
                for noise_freq, amplitude in zip(noise_freqs2, amplitudes2))
    signal_A = original_signal + noise_for_sigal_A 
    noisy_signal_B = signal_A + noise_for_sigal_B

    # Applying random shift
    # shift_samples = np.random.randint(-n // 2, n // 2)  # Random shift
    shift_samples = 3
    print(f"Shift Samples: {shift_samples}")
    signal_B = np.roll(noisy_signal_B, shift_samples)
    
    return signal_A, signal_B
signal_A, signal_B = generate_signals()



def dft(signal):
   
    N = len(signal)
    n = np.arange(N)
    k = n.reshape((N, 1))
    W = np.exp(-2j * np.pi * k * n / N)
    return np.dot(W, signal)

def idft(frequency_domain_signal):
    
    N = len(frequency_domain_signal)
    n = np.arange(N)
    k = n.reshape((N, 1))
    W_inv = np.exp(2j * np.pi * k * n / N) / N
    return np.dot(W_inv, frequency_domain_signal)



signal_A_dft = dft(signal_A)
signal_B_dft = dft(signal_B)


magnitude_A = np.abs(signal_A_dft)
magnitude_B = np.abs(signal_B_dft)



def cross_correlation(signal_A, signal_B):
    dft_A = dft(signal_A)
    dft_B = dft(signal_B)
    cross_corr_dft = dft_B * np.conjugate(dft_A) 
    cross_corr = idft(cross_corr_dft)
    return np.real(cross_corr)


cross_corr = cross_correlation(signal_A, signal_B)


lag_index = np.argmax(cross_corr)
if lag_index > n // 2:  
    lag_index -= n


lags = np.arange(-n // 2, n // 2)
cross_corr = np.roll(cross_corr, n // 2) 


time_lag = lag_index / sampling_rate 
distance = abs(time_lag * wave_velocity)  

print(f"Estimated Sample Lag: {lag_index}")
print(f"Estimated Time Lag: {time_lag:.4f} seconds")
print(f"Estimated Distance: {distance:.2f} meters")


def dft_filter(signal, threshold_ratio=0.7):

    signal_dft = dft(signal)
    magnitude = np.abs(signal_dft)
    threshold = threshold_ratio * np.max(magnitude)
    
    filtered_dft = np.where(magnitude > threshold, signal_dft, 0)
    

    filtered_signal = idft(filtered_dft)
    
    return filtered_signal, magnitude, filtered_dft


filtered_signal_A, magnitude_A, filtered_dft_A = dft_filter(signal_A)


filtered_signal_B, magnitude_B, filtered_dft_B = dft_filter(signal_B)

import os

output_dir = "task 1 plots"
os.makedirs(output_dir, exist_ok=True)

# Plot signal_A vs sample index
plt.figure(figsize=(10, 6))
plt.stem(samples, signal_A, linefmt="blue", markerfmt="bo", basefmt=' ')
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.title("Signal A (Station A)")
plt.legend()
plt.savefig(os.path.join(output_dir, "signal_A.png"))
plt.close()

# Plot signal_B vs sample index
plt.figure(figsize=(10, 6))
plt.stem(samples, signal_B, linefmt="red", markerfmt="ro", basefmt=' ')
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.title("Signal B (Station B)")
plt.legend()
plt.savefig(os.path.join(output_dir, "signal_B.png"))
plt.close()

# Plot frequency spectrum of Signal A
plt.figure(figsize=(10, 6))
plt.stem(samples, magnitude_A, linefmt="blue", markerfmt="bo", basefmt=' ')
plt.ylabel("Magnitude")
plt.title("Frequency Spectrum of Signal A")
plt.legend()
plt.savefig(os.path.join(output_dir, "frequency_spectrum_A.png"))
plt.close()

# Plot frequency spectrum of Signal B
plt.figure(figsize=(10, 6))
plt.stem(samples, magnitude_B, linefmt="red", markerfmt="ro", basefmt=' ')
plt.ylabel("Magnitude")
plt.title("Frequency Spectrum of Signal B")
plt.legend()
plt.savefig(os.path.join(output_dir, "frequency_spectrum_B.png"))
plt.close()

# Plot cross-correlation
plt.figure(figsize=(10, 6))
plt.stem(lags, cross_corr, linefmt="green", markerfmt="go", basefmt=' ')
plt.xlabel("Lag (samples)")
plt.ylabel("Cross-Correlation")
plt.title("DFT-based Cross-Correlation")
plt.savefig(os.path.join(output_dir, "cross_correlation.png"))
plt.close()

# Plot filtered Signal A
plt.figure(figsize=(10, 6))
plt.stem(samples, filtered_signal_A, linefmt="green", markerfmt="go", basefmt=' ', label="Filtered Signal A")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.title("DFT-Filtered Signal A")
plt.legend()
plt.savefig(os.path.join(output_dir, "filtered_signal_A.png"))
plt.close()

# Plot filtered Signal B
plt.figure(figsize=(10, 6))
plt.stem(samples, filtered_signal_B, linefmt="purple", markerfmt="mo", basefmt=' ', label="Filtered Signal B")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.title("DFT-Filtered Signal B")
plt.legend()
plt.savefig(os.path.join(output_dir, "filtered_signal_B.png"))
plt.close()





Shift Samples: 3
Estimated Sample Lag: 3
Estimated Time Lag: 0.0300 seconds
Estimated Distance: 240.00 meters


C:\Users\Asus\AppData\Local\Temp\ipykernel_14780\246520287.py:130: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
C:\Users\Asus\AppData\Local\Temp\ipykernel_14780\246520287.py:140: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
C:\Users\Asus\AppData\Local\Temp\ipykernel_14780\246520287.py:149: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
C:\Users\Asus\AppData\Local\Temp\ipykernel_14780\246520287.py:158: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.l